# Model Pipeline

Run Bayesian models across different input scalings and compare DIC.

**Output structure:** `data/models/{csv_stem}/{version}/`

In [3]:
import sys
sys.path.insert(0, '.')

import pandas as pd
from pyjags_pipeline import run_model, compare_dic, list_models

# Available input scalings
DATA_PATHS = {
    'per10k':    'data/wide_weekly_scaledPer10k.csv',
    'per-bed':   'data/wide_weekly_scaledPerBed.csv',
    'per-budget':'data/wide_weekly_scaledPerBudgetThousands.csv',
}

print('Available models:', list_models())

Available models: ['v0.1', 'v0.2', 'v2.1', 'v2.2', 'v2.3', 'v2.4', 'v2.5', 'v3.1', 'v3.2', 'v3.3', 'v4.1', 'v5.1']


## Run a single model at a single scaling

In [ ]:
result = run_model(
    version='v5.1',
    data_path=DATA_PATHS['per10k'],
    seed=42,
)

print('DIC:', result.dic)
print('Output:', result.output_dir)

## Run spine models at one scaling

In [ ]:
SPINE = ['v0.1', 'v2.1', 'v3.1', 'v4.1', 'v5.1']
SCALING = 'per10k'

for version in SPINE:
    run_model(version=version, data_path=DATA_PATHS[SCALING], seed=42)

## Run every available model at one scaling

In [ ]:
SCALING = 'per10k'

for version in list_models():
    run_model(version=version, data_path=DATA_PATHS[SCALING], seed=42)

## Run all models across all scalings

In [2]:
for scaling, path in DATA_PATHS.items():
    print(f'\n=== {scaling} ===')
    for version in list_models():
        run_model(version=version, data_path=path, seed=42)


=== per10k ===
[v0.1] Fitting AR(1) Only...
adapting: iterations 4000 of 4000, elapsed 0:00:00, remaining 0:00:00
sampling: iterations 40000 of 40000, elapsed 0:00:03, remaining 0:00:00
sampling: iterations 57656 of 80000, elapsed 0:00:06, remaining 0:00:02
sampling: iterations 80000 of 80000, elapsed 0:00:07, remaining 0:00:00
[v0.1] Computing DIC...
updating: iterations 59422 of 80000, elapsed 0:00:07, remaining 0:00:03
updating: iterations 80000 of 80000, elapsed 0:00:08, remaining 0:00:00
[v0.1] Computing Gelman-Rubin...
[v0.1] Reconstructing mu...
[v0.1] Computing fitted values and residuals...
[v0.1] Outputs exported to /Users/fluffypony/Library/Mobile Documents/com~apple~CloudDocs/Documents/Galway/PROJECT/CODE/data/models/wide_weekly_scaledPer10k/v0.1
[v0.1] Plots saved to /Users/fluffypony/Library/Mobile Documents/com~apple~CloudDocs/Documents/Galway/PROJECT/CODE/data/models/wide_weekly_scaledPer10k/v0.1/plots
[v0.1] Significance tests saved to /Users/fluffypony/Library/Mobile

## DIC comparison — per scaling

In [4]:
for scaling, path in DATA_PATHS.items():
    print(f'\n=== DIC table: {scaling} ===')
    try:
        df = compare_dic(path)
        display(df)
    except FileNotFoundError as e:
        print(f'  No results yet: {e}')


=== DIC table: per10k ===


,version,deviance,penalty,DIC
0,v4.1,4555.945342,93.596675,4649.542018
1,v5.1,4556.158956,94.104901,4650.263856
2,v3.2,4604.591347,87.327893,4691.919240
3,v3.1,4682.655557,57.527627,4740.183185
4,v3.3,5259.243969,57.169625,5316.413594
5,v2.1,5294.911046,51.151899,5346.062945
6,v0.2,5331.512090,28.621329,5360.133419
7,v2.5,5284.875853,75.338452,5360.214305
8,v0.1,5344.076309,26.547749,5370.624058
9,v2.4,5301.272522,75.803167,5377.075690



=== DIC table: per-bed ===


,version,deviance,penalty,DIC
0,v4.1,11448.892107,93.010592,11541.902699
1,v5.1,11449.025871,93.330444,11542.356315
2,v3.2,11470.035278,87.312203,11557.347481
3,v3.1,11510.104217,57.206986,11567.311202
4,v3.3,11738.612170,56.674637,11795.286807
5,v2.1,11756.672773,50.978073,11807.650846
6,v2.5,11751.828971,75.084209,11826.913181
7,v2.4,11773.252000,75.313080,11848.565081
8,v0.1,11828.673637,26.609311,11855.282948
9,v0.2,11828.277933,28.566858,11856.844791



=== DIC table: per-budget ===


,version,deviance,penalty,DIC
0,v5.1,19858.299038,80.591564,19938.890603
1,v3.1,19920.314654,54.940446,19975.255100
2,v4.1,19899.882216,79.377273,19979.259489
3,v3.2,19908.341599,77.369883,19985.711481
4,v3.3,20169.486862,50.389475,20219.876337
5,v2.1,20176.010662,48.582501,20224.593163
6,v2.5,20171.444711,72.073717,20243.518428
7,v2.4,20195.392089,70.605745,20265.997834
8,v0.1,20253.893406,27.000602,20280.894009
9,v0.2,20253.948705,29.293102,20283.241807


## Alpha (baseline intercept) — per scaling

In [ ]:
from pathlib import Path
import numpy as np

def _alpha_table(data_path, versions=None):
    stem = Path(data_path).stem
    scale_dir = Path('data/models') / stem
    if not scale_dir.exists():
        raise FileNotFoundError(f'No output for {stem}')
    frames = []
    for samples_path in sorted(scale_dir.glob('*/raw_samples.csv')):
        version = samples_path.parent.name
        if versions is not None and version not in versions:
            continue
        fitted_path = samples_path.parent / 'fitted.csv'
        if not fitted_path.exists():
            continue
        regions = list(pd.read_csv(fitted_path).columns)
        df = pd.read_csv(samples_path)
        alpha_cols = [c for c in df.columns if c.startswith('alpha[')]
        if not alpha_cols:
            continue
        rows = []
        for i, col in enumerate(sorted(alpha_cols, key=lambda x: int(x.strip('alpha[]')))):
            region = regions[i] if i < len(regions) else col
            s = df[col]
            rows.append({'Region': region,
                         'Mean': round(s.mean(), 4),
                         'SD': round(s.std(), 4),
                         '2.5%': round(np.percentile(s, 2.5), 4),
                         '97.5%': round(np.percentile(s, 97.5), 4)})
        tbl = pd.DataFrame(rows).set_index('Region')
        tbl.name = version
        frames.append((version, tbl))
    return frames

for scaling, path in DATA_PATHS.items():
    print(f'\n=== Alpha table: {scaling} ===')
    try:
        for version, tbl in _alpha_table(path, versions=['v5.1']):
            print(f'  -- {version}')
            display(tbl)
    except FileNotFoundError as e:
        print(f'  No results yet: {e}')



=== Alpha table: per10k ===
  -- v0.1


,Mean,SD,2.5%,97.5%
Region,,,,
HSE Dublin and Midlands,3.7960,0.2154,3.3674,4.2178
HSE Dublin and North East,2.1610,0.1402,1.8861,2.4366
HSE Dublin and South East,2.8500,0.1596,2.5331,3.1641
HSE Mid West,7.9051,0.3425,7.2234,8.5674
HSE South West,4.2356,0.2575,3.7241,4.7361
HSE West and North West,6.1433,0.3017,5.5460,6.7348


  -- v0.2


,Mean,SD,2.5%,97.5%
Region,,,,
HSE Dublin and Midlands,3.7896,0.2371,3.3176,4.2535
HSE Dublin and North East,2.1670,0.1569,1.8576,2.4733
HSE Dublin and South East,2.8577,0.1768,2.5072,3.2046
HSE Mid West,7.8384,0.3852,7.0668,8.5860
HSE South West,4.2335,0.2849,3.6670,4.7883
HSE West and North West,6.1373,0.3327,5.4671,6.7836


  -- v2.1


...

  -- v4.1


,Mean,SD,2.5%,97.5%
Region,,,,
HSE Dublin and Midlands,3.9933,0.1925,3.6129,4.3691
HSE Dublin and North East,2.2898,0.1332,2.0271,2.5518
HSE Dublin and South East,2.9954,0.1470,2.7089,3.2850
HSE Mid West,8.2451,0.3053,7.6410,8.8405
HSE South West,4.4224,0.2451,3.9395,4.9024
HSE West and North West,6.3620,0.2678,5.8333,6.8891


  -- v5.1


,Mean,SD,2.5%,97.5%
Region,,,,
HSE Dublin and Midlands,4.0007,0.1938,3.6214,4.3826
HSE Dublin and North East,2.2979,0.1329,2.0375,2.5595
HSE Dublin and South East,3.0018,0.1471,2.7124,3.2928
HSE Mid West,8.1810,0.3066,7.5701,8.7764
HSE South West,4.4247,0.2440,3.9424,4.9047
HSE West and North West,6.3390,0.2668,5.8134,6.8621



=== Alpha table: per-bed ===


## Ranks (alpha posterior ranks) — per scaling

In [ ]:
def _ranks_table(data_path, versions=None):
    stem = Path(data_path).stem
    scale_dir = Path('data/models') / stem
    if not scale_dir.exists():
        raise FileNotFoundError(f'No output for {stem}')
    frames = []
    for ranks_path in sorted(scale_dir.glob('*/ranks.csv')):
        version = ranks_path.parent.name
        if versions is not None and version not in versions:
            continue
        df = pd.read_csv(ranks_path, index_col=0)
        df.index.name = 'Region'
        df = df.rename(columns={'50%': 'Median'}).sort_values('ranked_mean')
        frames.append((version, df))
    return frames

for scaling, path in DATA_PATHS.items():
    print(f'\n=== Ranks table: {scaling} ===')
    try:
        for version, tbl in _ranks_table(path, versions=['v5.1']):
            print(f'  -- {version}')
            display(tbl)
    except FileNotFoundError as e:
        print(f'  No results yet: {e}')


SyntaxError: unexpected character after line continuation character (1938995583.py, line 5)

## Phi (AR(1) coefficient) — per scaling

In [7]:
def _phi_table(data_path, versions=None):
    stem = Path(data_path).stem
    scale_dir = Path('data/models') / stem
    if not scale_dir.exists():
        raise FileNotFoundError(f'No output for {stem\!r}')
    rows = []
    for gp_path in sorted(scale_dir.glob('*/global_parameters.csv')):
        version = gp_path.parent.name
        if versions is not None and version not in versions:
            continue
        df = pd.read_csv(gp_path)
        phi_rows = df[df['Parameter'].str.startswith('phi')]
        if phi_rows.empty:
            continue
        for _, row in phi_rows.iterrows():
            rows.append({'version': version, **row.to_dict()})
    if not rows:
        raise ValueError(f'No phi parameters found under {scale_dir}')
    return pd.DataFrame(rows).sort_values(['version', 'Parameter']).reset_index(drop=True)

for scaling, path in DATA_PATHS.items():
    print(f'\n=== Phi table: {scaling} ===')
    try:
        display(_phi_table(path, versions=['v5.1']))
    except (FileNotFoundError, ValueError) as e:
        print(f'  No results yet: {e}')


SyntaxError: unexpected character after line continuation character (1708895674.py, line 5)